# Study 956 — The Custody Fee 🏦

**The depositary bank charges you 1-5 cents per ADS per year and never sends a bill. Can the
tape see it?**

An American Depositary Receipt is a claim on foreign shares held by a depositary bank. The
bank charges the *holder* a pass-through custody fee — published in the deposit agreement,
typically **1-5 cents per ADS per year** — and nets
it out of a dividend rather than billing it. The foreign tax authority withholds tax on the
same dividend. Neither leak ever appears on a price chart.

We test it on **15 ADR / home-line pairs** across ten countries,
2000-01-03 → 2026-06-30 (6,649 rows), using daily **total-return** and
**price-only** closes.

*Real-tape numbers below are the frozen headline (`docs/results.md`, fingerprint
`ccd87508f03e`); the live cells run the fast offline synthetic control and are labelled as such.
As-of 2026-06-30.*


## 1. The bill nobody sends you

When you buy **TM** you are not buying a Toyota share. You are buying a receipt that a bank in New York issues against Toyota shares it holds in Tokyo. The bank does real work — collecting the dividend, converting the yen, handling the tax paperwork — and it charges you for it. A few cents per receipt per year, taken quietly out of the dividend before it lands in your account.

You will never see it on a chart, because the *price* of the receipt tracks the share almost perfectly. It only shows up in what you are actually **paid**.

> 🔬 *For the quants:* the estimand is the slope in time of `log(ADR total return) − log(home line total return × FX)`. Daily differences are hopeless here — non-synchronous closes put 1-2 % of noise a day around a fee worth 0.1 % a *year* — but that noise is stationary and reverses, so it does not accumulate, while the fee does. A trend fit on the level sees it; a mean of returns never will.

## 2. First, the trap — London has no dividends

We started with fifteen pairs. Five of them — Shell, BP, HSBC, Unilever, Rio Tinto — are London listings, and the naive comparison said their ADRs were bleeding **5.4 % a year**. A hundred times any real fee.

The reason is not finance, it is plumbing: the data vendor's "adjusted close" for the London Stock Exchange is **split-adjusted only**. The home leg's entire dividend stream is missing. Its measured yield is **0.04-0.05 %/yr** against the ADR's **3.58-5.82 %** — a ratio of **97-114×**.

So we built a screen that reads only the *home* line's own yield — never the ADR's, never the gap — and it throws all five out automatically. **10 of 15 pairs survive.** Anyone who skips this step will publish the 5 % number as a discovery.

In [1]:
R = {'uk_home': 0.05, 'uk_adr': 5.82, 'uk_ratio': 114, 'fake': -5.4, 'kept': 10, 'pairs': 15, 'gap_mean': 13.8, 'gap_median': 7.53, 'gap_pos': 9, 'gap_n': 10, 'sign_p': 0.0107, 'cents_median': 5.34, 'cents_ci_lo': 3.09, 'cents_ci_hi': 17.57}
print(f"London home leg yield : {R['uk_home']:.2f} %/yr   <- dividends missing entirely")
print(f"London ADR  leg yield : {R['uk_adr']:.2f} %/yr")
print(f"ratio                 : {R['uk_ratio']} x  ->  a fabricated "
      f"{R['fake']:.1f} %/yr 'custody fee'")
print(f"pairs surviving the coverage screen: {R['kept']} of {R['pairs']}")

London home leg yield : 0.05 %/yr   <- dividends missing entirely
London ADR  leg yield : 5.82 %/yr
ratio                 : 114 x  ->  a fabricated -5.4 %/yr 'custody fee'
pairs surviving the coverage screen: 10 of 15


## 3. On the ten pairs that work, something is there

Across the surviving ten, the ADR hands its holder **13.8 basis points a year** less income than the home line — median **7.5 bp**. In the unit the depositaries publish, that is a median of **5.3 cents per ADS per year**, which lands right on their stated **1-5 cent** band. Hold that thought — section 5 explains why landing on the band is *not* the same as having measured the fee.

**9 of 10** names point the same way — a coin would do that about once in a hundred tries (*p* = 0.011). Resampling the issuers puts the average at **[5.0, 25.6] bp/yr**, clear of zero.

In [2]:
names = [('TTE', 5.1, 1.58), ('SNY', -0.8, -0.42), ('SAP', 10.0, 1.78), ('PHG', 1.9, 2.65), ('ING', 4.4, 2.81), ('E', 31.6, 6.83), ('NVS', 59.3, 18.26), ('NVO', 1.5, 12.1), ('TM', 11.9, 7.69), ('TSM', 13.2, 13.62)]
print(f"{'name':<6} {'gap bp/yr':>10} {'HAC t':>8}")
for n, g, t in names:
    print(f"{n:<6} {g:>10.1f} {t:>8.2f}")
print()
print(f"pooled mean {R['gap_mean']:+.2f} bp/yr, median {R['gap_median']:+.2f}, "
      f"sign test {R['gap_pos']}/{R['gap_n']} positive (p = {R['sign_p']:.3f})")
print(f"in cents per ADS per year: median {R['cents_median']:.2f} c, "
      f"95% CI [{R['cents_ci_lo']:.2f}, {R['cents_ci_hi']:.2f}]")
print('published depositary schedules: 1-5 c/ADS/yr')

name    gap bp/yr    HAC t
TTE           5.1     1.58
SNY          -0.8    -0.42
SAP          10.0     1.78
PHG           1.9     2.65
ING           4.4     2.81
E            31.6     6.83
NVS          59.3    18.26
NVO           1.5    12.10
TM           11.9     7.69
TSM          13.2    13.62

pooled mean +13.80 bp/yr, median +7.53, sign test 9/10 positive (p = 0.011)
in cents per ADS per year: median 5.34 c, 95% CI [3.09, 17.57]
published depositary schedules: 1-5 c/ADS/yr


## 4. But it is a whisper, not a shout

Honesty first. The pooled *t* is **+2.36** — barely over the desk's bar — and dropping a single name (Eni) takes it to **+1.92**, under it. Worse, the ten names are not ten independent facts: six of them are euro-area issuers sharing one currency, one treaty rate and one dividend calendar. Count each currency block once and the *t* is **+1.84** on 5 observations.

Name by name the leak is invisible: a block bootstrap clears zero on only **3 of 10** issuers. And the average is propped up by Novartis, whose +59 bp/yr is really two spin-offs (Alcon 2019, Sandoz 2023) that the two data feeds record differently — drop it and the mean halves to **+8.8 bp/yr**.

It is also shrinking. Split the sample in 2015: **20.0 bp/yr** before, **10.2 bp/yr** since. Both positive — but half the size, and only 6 of 10 names still point the right way.

One more thing we cannot fix: every issuer here is still listed on both venues in 2026. That is a **survivor panel**, and the ADRs where fee disputes actually end up — the de-sponsored ones — are absent by construction.

## 5. And we cannot tell fee from tax

The plan was to split the leak in two: the depositary's fee, and the foreign government's withholding tax. It failed, and the failure is informative.

At the treaty rates, withholding alone should cost **26-96 bp/yr** on these dividend yields — three to seven times the *entire* gap we measure. Subtract it and every single name's "custody fee" goes **negative** (-43.1 bp/yr on average). That is not a finding about fees; it is proof that the withholding tax **is not in the total-return series at all** — the per-ADS dividend the data vendor records is the gross declared amount.

The fallback plan failed too. The UK charges *no* dividend withholding tax, so the five London pairs would have pinned the fee exactly — except those are the same five names section 2 threw out.

And now follow that first argument one step further, because it is the honest punchline of the study. If the vendor records the **gross declared** per-ADS dividend, then a fee the depositary bills straight to your brokerage account through DTC — which is how a great deal of the schedule is actually collected — never reaches this tape either. So **13.8 bp/yr is an upper bound on the depositary fee, not a measurement of it**. It is a combined income shortfall that is also consistent with the depositary's FX-conversion spread, with rounding of the per-ADS rate, and with feed differences on special dividends. Landing on the published 1-5 cent band is suggestive. It is not proof. **This study measures a leak; it does not name it.**

## 6. Live check — the machinery is unbiased (offline synthetic)

Below we build a world where we *know* the answer: a home line, an FX cross, and an ADR whose dividend is docked by a planted fee and a planted tax. The estimator must recover the planted number, and must report ~zero when we switch the fee off. **This cell is synthetic — none of its output is a real-tape result.**

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from adr_drag import data, strategy as st

frames, truth = data.synthetic_panel(n_names=8, drag_bps_per_year=25.0, signal_strength=1.0)
whts = {k: truth['per_name'][k]['wht'] for k in frames}
planted = st.synthetic_detect(frames, whts)
print('SYNTHETIC world with a planted fee')
print('  planted total leak  : %+.2f bp/yr' % (truth['planted_gap_per_year']*1e4))
print('  recovered           : %+.2f bp/yr' % (planted['income_gap']['mean']*1e4))
print('  residual custody    : %+.2f bp/yr (planted %.1f)'
      % (planted['custody']['mean']*1e4, truth['custody_drag_per_year']*1e4))

frames0, truth0 = data.synthetic_panel(n_names=8, drag_bps_per_year=25.0, signal_strength=0.0)
null = st.synthetic_detect(frames0, {k: 0.0 for k in frames0})
print('SYNTHETIC world with NO fee and NO tax')
print('  recovered           : %+.2f bp/yr  (should be ~0)' % (null['income_gap']['mean']*1e4))

SYNTHETIC world with a planted fee
  planted total leak  : +76.31 bp/yr
  recovered           : +76.44 bp/yr
  residual custody    : +24.45 bp/yr (planted 25.0)


SYNTHETIC world with NO fee and NO tax
  recovered           : +0.27 bp/yr  (should be ~0)


## Verdict

- **Signal — Weak.** An income shortfall is on the tape: **+13.8 bp/yr**, 9/10 names, sign test *p* = 0.011, a median of **5.3 cents per ADS/yr** in the range of the published schedules, positive in both eras. But *t* = +2.36 is knife-edge (leave-one-out +1.92; +1.84 once the euro names are counted once), a third of the intended sample was destroyed by a vendor data defect, the panel is a survivor panel, and the leak cannot be attributed — not to the tax, and not to the fee.
- **Tradability — Mirage.** Owning the home lines instead wins **+14.6 bp/yr gross at *t* = +0.11**, and a 15 bp/yr foreign safekeeping charge flips it negative. Budget ~5-10 cents per ADS per year of invisible wrapper cost and get on with your life.